# Land Cover per Embankment Segment — River & Land Side (v5)

Produces **one row per segment** (4220 rows) for each of the four land cover classes,
matching the exact column schema of `segment_vegetation_statistics_10m.csv`.

Output: one Excel sheet per class + Summary sheet.


## 1. Config

In [55]:
import os, warnings, gc, time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize as rio_rasterize
from rasterio.crs import CRS
from scipy.ndimage import distance_transform_edt
from shapely.geometry import mapping
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

BASE = r"Z:/Papers/Green solution/Imagery"
CLS  = r"Z:/Papers/Green solution/New_classified"

BUFFER_FILE    = os.path.join(BASE, "emb_100m_segment_50m_buffer.shp")
RIVER_TOE_FILE = os.path.join(BASE, "Toe_river_new.shp")
LAND_TOE_FILE  = os.path.join(BASE, "Toe_land_new.shp")

LC_FILES = {
    "Crop":        os.path.join(CLS, "Crop_no_building.shp"),
    "River":       os.path.join(CLS, "river_no_building.shp"),
    "Aquaculture": os.path.join(CLS, "aquaculture_no_building.shp"),
    "Building":    os.path.join(CLS, "Building.shp"),
}

SEGMENT_ID_FIELD = "Segment"
TARGET_EPSG      = 32645
RASTER_RES       = 2.0        # metres per pixel

OUTPUT_XLSX = os.path.join(BASE, "landcover_50m_results.xlsx")
os.makedirs(BASE, exist_ok=True)
print("Config ready.")


Config ready.


## 2. Load & Reproject

In [56]:
def load_vec(path, name, epsg):
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        raise ValueError(f"{name}: no CRS.")
    if gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(epsg=epsg)
    gdf.geometry = gdf.geometry.make_valid()
    print(f"  {name:<18}: {len(gdf):>5} features")
    return gdf

buffer_gdf = load_vec(BUFFER_FILE,    "Buffer",    TARGET_EPSG)
toe_river  = load_vec(RIVER_TOE_FILE, "Toe_river", TARGET_EPSG)
toe_land   = load_vec(LAND_TOE_FILE,  "Toe_land",  TARGET_EPSG)
lc_gdfs    = {n: load_vec(p, n, TARGET_EPSG) for n, p in LC_FILES.items()}

if SEGMENT_ID_FIELD not in buffer_gdf.columns:
    raise ValueError(f"'{SEGMENT_ID_FIELD}' not found. Columns: {buffer_gdf.columns.tolist()}")

buffer_gdf = buffer_gdf.reset_index(drop=True)
buffer_gdf["_idx"] = buffer_gdf.index + 1   # 1-based; 0 = background in label raster
print(f"\nTotal segments: {len(buffer_gdf)}")
print("Segment IDs sample:", buffer_gdf[SEGMENT_ID_FIELD].values[:10])


  Buffer            :  4220 features
  Toe_river         :     1 features
  Toe_land          :    13 features
  Crop              : 118553 features
  River             : 90756 features
  Aquaculture       : 163668 features
  Building          : 207616 features

Total segments: 4220
Segment IDs sample: [ 1  2  3  4  5  6  7  8  9 10]


## 3. Build Global Raster Grid

In [57]:
minx, miny, maxx, maxy = buffer_gdf.total_bounds
pad = max(RASTER_RES * 10, 50)
minx -= pad; miny -= pad; maxx += pad; maxy += pad

width  = int(np.ceil((maxx - minx) / RASTER_RES))
height = int(np.ceil((maxy - miny) / RASTER_RES))
transform  = from_bounds(minx, miny, maxx, maxy, width, height)
cell_area_m2 = RASTER_RES ** 2

print(f"Grid     : {width:,} × {height:,} px  at {RASTER_RES} m/px")
print(f"Cell area: {cell_area_m2} m²")
print(f"Total px : {width*height:,}")


Grid     : 18,806 × 20,797 px  at 2.0 m/px
Cell area: 4.0 m²
Total px : 391,108,382


## 4. Burn Segment Label Raster (One Pass)

Every pixel gets the integer `_idx` of the segment it falls in (0 = outside all segments).
This replaces the per-segment `geometry_mask()` loop entirely.


In [58]:
print("Burning segment label raster ...")
t0 = time.time()

label_shapes = (
    (mapping(row.geometry), int(row["_idx"]))
    for _, row in buffer_gdf.iterrows()
    if row.geometry and not row.geometry.is_empty
)

label_arr = rio_rasterize(
    label_shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="int32",
    all_touched=False,
)

print(f"  Done in {time.time()-t0:.1f}s")
n_labelled  = (label_arr > 0).sum()
n_unique    = len(np.unique(label_arr[label_arr > 0]))
print(f"  Labelled pixels  : {n_labelled:,}")
print(f"  Unique segments  : {n_unique:,}  (expected {len(buffer_gdf)})")


Burning segment label raster ...
  Done in 1.9s
  Labelled pixels  : 10,367,695
  Unique segments  : 4,219  (expected 4220)


## 5. Toe Lines → Distance → Side Raster

In [59]:
def burn_lines(gdf):
    shapes = ((mapping(g), 1) for g in gdf.geometry if g and not g.is_empty)
    return rio_rasterize(shapes, out_shape=(height, width),
                         transform=transform, fill=0, dtype="uint8", all_touched=True)

print("Rasterizing toe lines ...")
rv_toe = burn_lines(toe_river)
ld_toe = burn_lines(toe_land)
print(f"  River toe pixels: {rv_toe.sum():,}")
print(f"  Land  toe pixels: {ld_toe.sum():,}")

if rv_toe.sum() == 0:
    raise RuntimeError("River toe: 0 pixels — check CRS / extent overlap.")
if ld_toe.sum() == 0:
    raise RuntimeError("Land toe: 0 pixels — check CRS / extent overlap.")

print("Computing distance transforms ...")
t0 = time.time()
dist_rv = distance_transform_edt(rv_toe == 0) * RASTER_RES
dist_ld = distance_transform_edt(ld_toe == 0) * RASTER_RES
print(f"  Done in {time.time()-t0:.1f}s")

# 1 = river side, 2 = land side
side_arr = np.where(dist_rv < dist_ld, np.uint8(1),
           np.where(dist_ld < dist_rv, np.uint8(2), np.uint8(0)))

del dist_rv, dist_ld, rv_toe, ld_toe
gc.collect()

print(f"  River-side pixels : {(side_arr==1).sum():,}")
print(f"  Land-side  pixels : {(side_arr==2).sum():,}")


Rasterizing toe lines ...
  River toe pixels: 265,956
  Land  toe pixels: 266,138
Computing distance transforms ...
  Done in 68.9s
  River-side pixels : 180,174,625
  Land-side  pixels : 210,876,440


## 6. Rasterize Land Cover Layers

In [60]:
def burn_polys(gdf):
    shapes = ((mapping(g), 1) for g in gdf.geometry if g and not g.is_empty)
    return rio_rasterize(shapes, out_shape=(height, width),
                         transform=transform, fill=0, dtype="uint8", all_touched=False)

print("Rasterizing land cover ...")
lc_arrs = {}
for name, gdf in lc_gdfs.items():
    t0 = time.time()
    arr = burn_polys(gdf)
    lc_arrs[name] = arr
    print(f"  {name:<14}: {arr.sum():>10,} px  ({time.time()-t0:.1f}s)")


Rasterizing land cover ...
  Crop          : 79,943,937 px  (112.3s)
  River         : 44,526,882 px  (24.1s)
  Aquaculture   : 92,749,132 px  (70.2s)
  Building      :  2,299,222 px  (18.5s)


## 7. Flatten to Pixel Table

Keep only pixels inside a segment buffer.
One row per pixel, columns: `_idx`, `side`, `Crop`, `River`, `Aquaculture`, `Building`.


In [61]:
print("Building flat pixel table ...")
t0 = time.time()

in_seg = label_arr > 0   # mask: only segment pixels

flat = {
    "_idx": label_arr[in_seg].astype(np.int32),
    "side": side_arr[in_seg].astype(np.int8),
}
for name, arr in lc_arrs.items():
    flat[name] = arr[in_seg].astype(np.int8)

df_px = pd.DataFrame(flat)
df_px["is_rv"] = (df_px["side"] == 1)
df_px["is_ld"] = (df_px["side"] == 2)

print(f"  Pixel rows   : {len(df_px):,}")
print(f"  Done in {time.time()-t0:.1f}s")
print(df_px.head(3))


Building flat pixel table ...
  Pixel rows   : 10,367,695
  Done in 1.2s
   _idx  side  Crop  River  Aquaculture  Building  is_rv  is_ld
0  4220     1     0      0            0         0   True  False
1  4220     1     0      0            0         0   True  False
2  4220     1     0      0            0         0   True  False


## 8. Aggregate Per Segment — One Row Per Segment

**This is the fix.** We pre-aggregate ALL counts in one `groupby` call,
then do the percentage math in a fully vectorised way — no Python loop over segments,
no `.apply()` that collapses results.


In [62]:
print("Aggregating pixel counts per segment ...")
t0 = time.time()

lc_names = list(lc_arrs.keys())

# ── Step 1: build all aggregation columns in ONE groupby ─────────────────
#  For each LC class we need:
#    total class pixels in segment
#    class pixels on river side
#    class pixels on land side
#  Plus: total pixels, river-side pixels, land-side pixels

agg_dict = {
    "total_px": ("side", "count"),      # all pixels in segment
    "rv_px":    ("is_rv", "sum"),        # river-side pixels
    "ld_px":    ("is_ld", "sum"),        # land-side pixels
}
for name in lc_names:
    agg_dict[f"{name}_total"] = (name, "sum")

# River-side and land-side LC counts need extra boolean columns
for name in lc_names:
    df_px[f"{name}_rv"] = (df_px[name].astype(bool) & df_px["is_rv"]).astype(np.int8)
    df_px[f"{name}_ld"] = (df_px[name].astype(bool) & df_px["is_ld"]).astype(np.int8)
    agg_dict[f"{name}_rv_px"] = (f"{name}_rv", "sum")
    agg_dict[f"{name}_ld_px"] = (f"{name}_ld", "sum")

grp = df_px.groupby("_idx").agg(**agg_dict).reset_index()

print(f"  Aggregation done in {time.time()-t0:.1f}s")
print(f"  Rows in aggregated table: {len(grp)}  (expected {len(buffer_gdf)})")
print(grp.head(3).to_string())


Aggregating pixel counts per segment ...
  Aggregation done in 1.1s
  Rows in aggregated table: 4219  (expected 4220)
   _idx  total_px  rv_px  ld_px  Crop_total  River_total  Aquaculture_total  Building_total  Crop_rv_px  Crop_ld_px  River_rv_px  River_ld_px  Aquaculture_rv_px  Aquaculture_ld_px  Building_rv_px  Building_ld_px
0     1      1786    920    863         324            2                805              58           0         324            2            0                799                  5               0              58
1     2      1274    685    580           0          648                626               0           0           0          588           60                 97                520               0               0
2     3      2040   1017   1006         400          305               1155               0         237         153           58          247                636                516               0               0


## 9. Compute Statistics & Build Per-Segment DataFrames

Vectorised pandas math — no Python loop over segments.


In [63]:
# Merge aggregated pixel counts with the segment ID lookup
idx_to_seg = buffer_gdf.set_index("_idx")[SEGMENT_ID_FIELD]
grp[SEGMENT_ID_FIELD] = grp["_idx"].map(idx_to_seg)

# Segments with zero raster pixels get all-zero rows
# (these are segments whose geometry didn't cover even one pixel)
missing_segs = buffer_gdf[~buffer_gdf["_idx"].isin(grp["_idx"])][SEGMENT_ID_FIELD].values
if len(missing_segs) > 0:
    print(f"WARNING: {len(missing_segs)} segments had 0 raster pixels — they will appear with zeros.")

# ── Convert pixel counts → areas ─────────────────────────────────────────
grp["valid_a"] = grp["total_px"] * cell_area_m2
grp["rv_a"]    = grp["rv_px"]    * cell_area_m2
grp["ld_a"]    = grp["ld_px"]    * cell_area_m2

for name in lc_names:
    grp[f"{name}_veg_a"]    = grp[f"{name}_total"] * cell_area_m2
    grp[f"{name}_rv_veg_a"] = grp[f"{name}_rv_px"] * cell_area_m2
    grp[f"{name}_ld_veg_a"] = grp[f"{name}_ld_px"] * cell_area_m2

# ── Build one DataFrame per LC class ─────────────────────────────────────
COLUMN_ORDER = [
    SEGMENT_ID_FIELD,
    "Veg_present","Valid_area_m2","Veg_area_m2","Veg_pct",
    "River_veg_present","River_valid_area_m2","River_veg_area_m2","River_veg_pct",
    "Land_veg_present","Land_valid_area_m2","Land_veg_area_m2","Land_veg_pct",
    "River_share_total_veg_pct","Land_share_total_veg_pct",
    "Tree_side","Dominant_veg_side",
]

lc_dfs = {}

for name in lc_names:
    g = grp.copy()

    veg_a    = g[f"{name}_veg_a"]
    rv_veg_a = g[f"{name}_rv_veg_a"]
    ld_veg_a = g[f"{name}_ld_veg_a"]
    valid_a  = g["valid_a"]
    rv_a     = g["rv_a"]
    ld_a     = g["ld_a"]

    # Percentages — vectorised, avoid /0
    veg_pct = np.where(valid_a > 0, veg_a / valid_a * 100, np.nan)
    rv_pct  = np.where(rv_a    > 0, rv_veg_a / rv_a * 100, np.nan)
    ld_pct  = np.where(ld_a    > 0, ld_veg_a / ld_a * 100, np.nan)
    rv_shr  = np.where(veg_a   > 0, rv_veg_a / veg_a * 100, 0.0)
    ld_shr  = np.where(veg_a   > 0, ld_veg_a / veg_a * 100, 0.0)

    # Tree_side — vectorised using np.select
    ts_cond = [
        (rv_veg_a > 0) & (ld_veg_a > 0),
        (rv_veg_a > 0) & (ld_veg_a == 0),
        (rv_veg_a == 0) & (ld_veg_a > 0),
    ]
    tree_side = np.select(ts_cond, ["Both", "River", "Land"], default="None")

    # Dominant_veg_side — vectorised
    dom_cond = [
        veg_a == 0,
        np.isnan(rv_pct) & np.isnan(ld_pct),
        np.isnan(rv_pct),
        np.isnan(ld_pct),
        rv_pct > ld_pct,
        ld_pct > rv_pct,
        rv_pct == ld_pct,
    ]
    dom_side = np.select(dom_cond,
                         ["None","None","Land","River","River","Land","Equal"],
                         default="None")

    # Handle NaN case for Dominant when veg_area is 0
    dom_side = np.where(veg_a == 0, "None", dom_side)

    df = pd.DataFrame({
        SEGMENT_ID_FIELD:            g[SEGMENT_ID_FIELD].values,
        "Veg_present":               (veg_a > 0).astype(int),
        "Valid_area_m2":             valid_a.round(2),
        "Veg_area_m2":               veg_a.round(2),
        "Veg_pct":                   np.where(np.isnan(veg_pct), np.nan, veg_pct.round(2)),
        "River_veg_present":         (rv_veg_a > 0).astype(int),
        "River_valid_area_m2":       rv_a.round(2),
        "River_veg_area_m2":         rv_veg_a.round(2),
        "River_veg_pct":             np.where(np.isnan(rv_pct), np.nan, rv_pct.round(2)),
        "Land_veg_present":          (ld_veg_a > 0).astype(int),
        "Land_valid_area_m2":        ld_a.round(2),
        "Land_veg_area_m2":          ld_veg_a.round(2),
        "Land_veg_pct":              np.where(np.isnan(ld_pct), np.nan, ld_pct.round(2)),
        "River_share_total_veg_pct": rv_shr.round(2),
        "Land_share_total_veg_pct":  ld_shr.round(2),
        "Tree_side":                 tree_side,
        "Dominant_veg_side":         dom_side,
    })

    # Add any segments that had 0 raster pixels
    if len(missing_segs) > 0:
        empty_rows = pd.DataFrame([{
            SEGMENT_ID_FIELD: s,
            "Veg_present": 0, "Valid_area_m2": 0, "Veg_area_m2": 0, "Veg_pct": np.nan,
            "River_veg_present": 0, "River_valid_area_m2": 0,
            "River_veg_area_m2": 0, "River_veg_pct": np.nan,
            "Land_veg_present": 0, "Land_valid_area_m2": 0,
            "Land_veg_area_m2": 0, "Land_veg_pct": np.nan,
            "River_share_total_veg_pct": 0, "Land_share_total_veg_pct": 0,
            "Tree_side": "None", "Dominant_veg_side": "None",
        } for s in missing_segs])
        df = pd.concat([df, empty_rows], ignore_index=True)

    df = df.sort_values(SEGMENT_ID_FIELD).reset_index(drop=True)
    lc_dfs[name] = df
    print(f"  {name:<14}: {len(df)} rows  |  Veg_present={df['Veg_present'].sum()}")

print(f"\nAll {len(lc_dfs)} LC classes processed.")


  Crop          : 4220 rows  |  Veg_present=3648
  River         : 4220 rows  |  Veg_present=3730
  Aquaculture   : 4220 rows  |  Veg_present=3998
  Building      : 4220 rows  |  Veg_present=3342

All 4 LC classes processed.


## 10. Verify — Must Be One Row Per Segment

In [64]:
expected = len(buffer_gdf)
all_ok = True

for name, df in lc_dfs.items():
    n = len(df)
    dups = df[SEGMENT_ID_FIELD].duplicated().sum()
    status = "OK" if (n == expected and dups == 0) else "PROBLEM"
    if status == "PROBLEM":
        all_ok = False
    print(f"  {name:<14}: {n} rows  |  duplicates: {dups}  |  [{status}]")

print()
if all_ok:
    print(f"All classes have exactly {expected} rows — one per segment.")
else:
    print("WARNING: row count mismatch — check above.")

# Sample output
print("\nSample (Crop):")
print(lc_dfs['Crop'].head(10).to_string(index=False))


  Crop          : 4220 rows  |  duplicates: 0  |  [OK]
  River         : 4220 rows  |  duplicates: 0  |  [OK]
  Aquaculture   : 4220 rows  |  duplicates: 0  |  [OK]
  Building      : 4220 rows  |  duplicates: 0  |  [OK]

All classes have exactly 4220 rows — one per segment.

Sample (Crop):
 Segment  Veg_present  Valid_area_m2  Veg_area_m2  Veg_pct  River_veg_present  River_valid_area_m2  River_veg_area_m2  River_veg_pct  Land_veg_present  Land_valid_area_m2  Land_veg_area_m2  Land_veg_pct  River_share_total_veg_pct  Land_share_total_veg_pct Tree_side Dominant_veg_side
       1            1         7144.0       1296.0    18.14                  0               3680.0                0.0           0.00                 1              3452.0            1296.0         37.54                       0.00                    100.00      Land              Land
       2            0         5096.0          0.0     0.00                  0               2740.0                0.0           0.00         

## 11. Save Excel — One Sheet per Class

In [65]:
summary_rows = []
for lc_name, df in lc_dfs.items():
    summary_rows.append({
        "Land_Cover":             lc_name,
        "Total_segments":         len(df),
        "Segments_with_class":    int(df["Veg_present"].sum()),
        "Segments_without_class": int((df["Veg_present"]==0).sum()),
        "Mean_Veg_pct":           round(df["Veg_pct"].mean(), 2),
        "Mean_River_veg_pct":     round(df["River_veg_pct"].mean(), 2),
        "Mean_Land_veg_pct":      round(df["Land_veg_pct"].mean(), 2),
        "Mean_River_share_pct":   round(df["River_share_total_veg_pct"].mean(), 2),
        "Mean_Land_share_pct":    round(df["Land_share_total_veg_pct"].mean(), 2),
        "Tree_Both":   int((df["Tree_side"]=="Both").sum()),
        "Tree_River":  int((df["Tree_side"]=="River").sum()),
        "Tree_Land":   int((df["Tree_side"]=="Land").sum()),
        "Tree_None":   int((df["Tree_side"]=="None").sum()),
        "Dom_River":   int((df["Dominant_veg_side"]=="River").sum()),
        "Dom_Land":    int((df["Dominant_veg_side"]=="Land").sum()),
        "Dom_Equal":   int((df["Dominant_veg_side"]=="Equal").sum()),
        "Dom_None":    int((df["Dominant_veg_side"]=="None").sum()),
    })

summary_df = pd.DataFrame(summary_rows)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    for lc_name, df in lc_dfs.items():
        df.to_excel(writer, sheet_name=lc_name, index=False)
    combined = pd.concat(
        [df.assign(Land_Cover=n) for n, df in lc_dfs.items()],
        ignore_index=True
    )
    combined.to_excel(writer, sheet_name="All_Combined", index=False)

print(f"Saved: {OUTPUT_XLSX}")
print(f"Sheets: Summary | {' | '.join(lc_dfs.keys())} | All_Combined")
print()
print(summary_df.to_string(index=False))


Saved: Z:/Papers/Green solution/Imagery\landcover_50m_results.xlsx
Sheets: Summary | Crop | River | Aquaculture | Building | All_Combined

 Land_Cover  Total_segments  Segments_with_class  Segments_without_class  Mean_Veg_pct  Mean_River_veg_pct  Mean_Land_veg_pct  Mean_River_share_pct  Mean_Land_share_pct  Tree_Both  Tree_River  Tree_Land  Tree_None  Dom_River  Dom_Land  Dom_Equal  Dom_None
       Crop            4220                 3648                     572         35.45               32.13              38.76                 37.15                48.86       2737         320        591        572       1488      2156          4       572
      River            4220                 3730                     490         16.99               22.78              11.28                 55.49                32.88       2103        1105        522        490       2480      1250          0       490
Aquaculture            4220                 3998                     222         35.56       

## 12. Visualisation

In [ ]:
LC_COLORS = {"Crop":"#26de81","River":"#00d4ff","Aquaculture":"#a29bfe","Building":"#ff6b6b"}

fig, axes = plt.subplots(len(lc_dfs), 3, figsize=(20, 5*len(lc_dfs)), facecolor="#0d1117")
if len(lc_dfs) == 1: axes = [axes]
fig.suptitle("Land Cover % per Segment — River vs Land Side",
             color="white", fontsize=14, fontweight="bold")

pie_c = {"River":"#00d4ff","Land":"#ff9f43","Equal":"#a29bfe",
         "None":"#636e72","Both":"#26de81"}

for ri, (lc_name, df) in enumerate(lc_dfs.items()):
    df_s = df.sort_values(SEGMENT_ID_FIELD)
    x    = df_s[SEGMENT_ID_FIELD]

    ax0 = axes[ri][0]; ax0.set_facecolor("#0d1117")
    ax0.fill_between(x, df_s["River_veg_pct"].fillna(0), alpha=0.2, color="#00d4ff")
    ax0.plot(x, df_s["River_veg_pct"].fillna(0), color="#00d4ff", lw=1.5, label="River %")
    ax0.fill_between(x, df_s["Land_veg_pct"].fillna(0), alpha=0.2, color="#ff9f43")
    ax0.plot(x, df_s["Land_veg_pct"].fillna(0), color="#ff9f43", lw=1.5, ls="--", label="Land %")
    ax0.set_title(f"{lc_name} — % of side", color="white", fontsize=10, fontweight="bold")
    ax0.legend(fontsize=8, facecolor="#1a1a2e", labelcolor="white")
    ax0.set_ylabel("%", color="#aaa", fontsize=8)

    ax1 = axes[ri][1]; ax1.set_facecolor("#0d1117")
    w = 0.4; xi = np.arange(len(df_s))
    ax1.bar(xi-w/2, df_s["River_share_total_veg_pct"].fillna(0), w,
            color="#00d4ff", alpha=0.8, label="River share")
    ax1.bar(xi+w/2, df_s["Land_share_total_veg_pct"].fillna(0),  w,
            color="#ff9f43", alpha=0.8, label="Land share")
    ax1.set_title(f"{lc_name} — share of total", color="white", fontsize=10, fontweight="bold")
    ax1.legend(fontsize=8, facecolor="#1a1a2e", labelcolor="white")
    ax1.set_ylabel("% of total class area", color="#aaa", fontsize=8)

    ax2 = axes[ri][2]; ax2.set_facecolor("#0d1117")
    dom = df_s["Dominant_veg_side"].value_counts()
    ax2.pie(dom.values, labels=dom.index,
            colors=[pie_c.get(k,"#888") for k in dom.index],
            autopct="%1.0f%%",
            textprops={"color":"white","fontsize":9},
            wedgeprops={"edgecolor":"#0d1117","linewidth":1.5})
    ax2.set_title(f"{lc_name} — dominant side", color="white", fontsize=10, fontweight="bold")

    for ax in axes[ri]:
        ax.tick_params(colors="#aaa", labelsize=7)
        for sp in ax.spines.values(): sp.set_edgecolor("#333")
        ax.set_xlabel("Segment", color="#aaa", fontsize=7)

plt.tight_layout()
plot_path = OUTPUT_XLSX.replace(".xlsx","_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"Saved: {plot_path}")
